#CafChem tools for building a GPT model, or fine-tuning a GPT model to produce molecules based on a user-provided dataset

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MauricioCafiero/CafChem/blob/main/notebooks/GPT_CafChem.ipynb)

## This notebook allows you to:
- Read in a CSV file with SMILES strings and tokenize it.
- Build a **foundation** GPT from the ZN305K SMILES dataset (PyTorch).
- Train or fine-tune a GPT on this dataset.
- Save the GPT for future use.
- Upload a saved GPT.
- Test the GPT by generating sample molecules.
- Run large-scale inference to generate up to 5K molecules.

## Requirements:
- PyTorch (Apple Silicon MPS by default; falls back to CUDA then CPU).
- rdkit, pandas, numpy, scikit-learn, matplotlib.


## Set-up

This block:

- Installs the needed libraries.
- Adds the local `code/` directory to the path and imports the PyTorch CafChem tools.
- Apple Silicon (MPS) is selected automatically; CUDA and CPU are fallbacks.


In [ ]:
!pip install torch rdkit pandas numpy scikit-learn matplotlib

In [ ]:
import sys, os
# If running locally, point at the repo's code/ directory.
# In Colab, upload the code/ folder (or clone the CafChem repo) first.
for p in ("code", "../code", "CafChem/code"):
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)
print("sys.path:", sys.path[:3])

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rdkit.Chem import AllChem, Draw
import CafChemGPT as ccgpt

print("All libraries loaded!")
print("Device:", ccgpt.get_device())
print("Torch:", torch.__version__, "| MPS available:", torch.backends.mps.is_available())

## Build the foundation model (ZN305K)

Per the project instructions, a foundation GPT is built and saved from `data/ZN305K_smiles.csv`.
This is a 2-block model with the 100-token vocabulary (`data/vocab_305K.txt`) and a 166-token context window.
Training the full foundation takes a while; `FOUNDATION_EPOCHS` can be raised (the original used ~50).


In [ ]:
fx_f, fy_f, VOCAB_SIZE_f, tok_f, max_length_f = ccgpt.make_datasets("data/ZN305K_smiles.csv", "SMILES")
gpt_f = ccgpt.make_gpt(2, max_length_f, VOCAB_SIZE_f)
gpt_f = ccgpt.train_gpt(gpt_f, fx_f, fy_f, epochs=10, batch_size=512)
ccgpt.save_gpt(gpt_f, "data/GPT_ZN305_pytorch")  # saved as data/GPT_ZN305_pytorch.pt

## Tokenize
- Create the tokenizer (right now the vocabulary is limited to 100 tokens). You can test the compapibility of your dataset with the vocabulary below.
- Create the input and target arrays for training.

AttributeError: module 'tensorflow' has no attribute '__version'

In [ ]:
fx, fy, VOCAB_SIZE, tokenizer, max_length = ccgpt.make_datasets("Tyrosinase1239_IC50.csv", "SMILES")

In [ ]:
print(f"Vocabulary size: {VOCAB_SIZE}")
print(f"Max length: {max_length}")
print(f"fx: {fx.shape}, fy: {fy.shape}")

## Training the model

- This block defines the GPT (PyTorch `nn.Module`) and runs the optimization with `train_gpt`.
- `train_gpt` replaces the old Keras `compile`/`fit` pattern.


In [ ]:
gpt = ccgpt.make_gpt(2, max_length, VOCAB_SIZE)

In [ ]:
batch_size = 512
gpt = ccgpt.train_gpt(gpt, fx, fy, epochs=2, batch_size=batch_size)

## saving
- saves both the model weights and the layer names (in a txt file). These are needed for loading later.

In [ ]:
gpt.summary()

In [ ]:
ccgpt.save_gpt(gpt, "GPT_statin_2epochs")

## Set up Xfer learning
- first, upload a training set and test the vocabulary to see if your dataset is compatible with the Foundation model usedfor fine-tuning. It will check if you dataset has any token not included in the foundation model, and check to see if the context window for your dataset is within the context window for the foundation model.
- if it is compatible, tokenize your dataset.
- then create a new model that has 2 transformer blocks/layers pre-loaded from the foundation model, with a user-specified number of new blocks for fine-tuning. By default the old blocks are frozen for training, but you can toggle this.
- (optionally unfreeze the old blocks/layers)
- finally train the model. About 50 epochs of training minumum are usually required for fine-tuning. Model should be trained first with the foundation layers frozen, and then again with them unfrozen.
- The foundation model is a 2 transformer block model trained on 305K SMILES from the ZN database. Of those, at least 55K are drug-like.

In [ ]:
novel_tokens = ccgpt.test_vocab("Tyrosinase1239_IC50.csv", "SMILES")

In [ ]:
ccgpt.trim_vocab("Tyrosinase1239_IC50.csv", novel_tokens)

In [ ]:
fx, fy, VOCAB_SIZE, tokenizer, max_length = ccgpt.make_datasets("Tyrosinase1239_IC50_trimmed.csv", "SMILES")

In [ ]:
gpt_2x_2 = ccgpt.make_finetune_gpt(2, freeze_old_layers=True)

In [ ]:
gpt_2x_2 = ccgpt.unfreeze_gpt(gpt_2x_2)

In [ ]:
batch_size = 512
gpt_2x_2 = ccgpt.train_gpt(gpt_2x_2, fx, fy, epochs=75, batch_size=batch_size)

In [ ]:
ccgpt.save_gpt(gpt_2x_2, "GPT_statin_100p75epochs")

## Standard molecule prediction

This block

- Accepts a model and generates a sample of up to 5 molecules as a quick test for the model.
- you can load just the foundation model to use, or
- use any other model.


In [ ]:
gpt_found = ccgpt.load_foundation()

In [ ]:
pic = ccgpt.test_gen(gpt_found, tokenizer, 1.0, VOCAB_SIZE, 42)

In [ ]:
pic

## Inference
- Load any model: just provide the string of the model's filename, the total number of blocks, and the context window and vocabulary size (from a tokenization above).
- Create up to 5K prompts with a length up to the context window size. Prompt lengths less than 6 tokens are recommended.

In [ ]:
loaded_model = ccgpt.load_gpt("GPT_statin_100p75epochs", 4, max_length, VOCAB_SIZE)

In [ ]:
loaded_model = ccgpt.load_foundation()

In [ ]:
prompts = ccgpt.make_prompts(100, 2)

In [ ]:
pic, novel_smiles = ccgpt.gen_mols(prompts, True, loaded_model, tokenizer, 1.5, VOCAB_SIZE)

In [ ]:
pic

## Create specific vocab list

- First block must be run before using this block. Then swap out the old vocab file for the new one and rerun
first block.
- reads in original vocab list
- uses the fl2set from the first block: this is a set with all unique tokens used in this model
- if a token is used, it is added to the new vocab list
- Note: [unused] must be added after [PAD] if it is not included

In [ ]:
# Routine to create a smaller vocab file based on the tokens used in a dataset.
# Requires the token-id set `fl2set` computed from make_datasets (advanced/optional).
oldvocab = open("data/vocab.txt", "r")
oldvocab_lines = oldvocab.readlines()
oldvocab.close()

newvocab = open("data/vocab_350K.txt", "w", newline="\n")
for i in range(len(oldvocab_lines)):
    if i in fl2set:
        print(oldvocab_lines[i])
        newvocab.write(oldvocab_lines[i])
newvocab.close()

In [ ]:
rnd = random.